# JSON Data Analysis

This notebook was generated from JSON data using the JSON to IPYNB converter.


## Data from analysis.json

Analysis of the JSON data structure and content.


In [ ]:
import json
import pandas as pd
import numpy as np

# Load the JSON data
data = {
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Marmara University Göztepe Campus — Fiber-Optic Network Optimization\n",
        "## MIS Network Analysis Notebook\n",
        "\n",
        "This notebook walks through the Maximum Flow analysis **step by step** with\n",
        "explanatory commentary between each code cell.  \n",
        "For the production-grade script, see `src/solution.py`.\n",
        "\n",
        "---"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 0 · Imports and Setup"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "import os, csv\n",
        "import networkx as nx\n",
        "import matplotlib.pyplot as plt\n",
        "import pandas as pd\n",
        "\n",
        "%matplotlib inline\n",
        "plt.rcParams['figure.dpi'] = 120\n",
        "\n",
        "DATA_FILE = os.path.join('..', 'data', 'network_data.csv')\n",
        "print('Libraries loaded successfully.')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 1 · Explore the Dataset\n",
        "\n",
        "Before building the graph, we inspect the CSV to understand what each column represents.\n",
        "\n",
        "| Column | Description |\n",
        "|---|---|\n",
        "| `source` / `target` | Campus building IDs (nodes) |\n",
        "| `capacity_mbps` | Maximum fiber throughput — the **key constraint** for max-flow |\n",
        "| `distance_m` | Physical cable run length (metadata only) |\n",
        "| `cable_cost_tl` | Estimated installation cost in Turkish Lira (metadata only) |\n",
        "| `node_type_*` | Role of each building (core / admin / faculty / lab / service) |"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "df = pd.read_csv(DATA_FILE)\n",
        "print(f'Shape: {df.shape[0]} edges × {df.shape[1]} columns')\n",
        "df.head(14)"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Summary statistics — understand the capacity distribution\n",
        "print('Capacity distribution (Mbps):')\n",
        "print(df['capacity_mbps'].describe().astype(int).to_string())\n",
        "print(f'\\nTotal infrastructure cost: ₺{df[\"cable_cost_tl\"].sum():,}')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 2 · Build the Directed Graph\n",
        "\n",
        "Each CSV row becomes **two directed arcs** (forward + reverse) to model\n",
        "the bidirectional nature of fiber-optic cable.\n",
        "\n",
        "The `capacity` attribute on each arc is what the Maximum Flow solver will\n",
        "respect as an upper bound."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "G = nx.DiGraph()\n",
        "\n",
        "with open(DATA_FILE, newline='', encoding='utf-8') as f:\n",
        "    reader = csv.DictReader(f)\n",
        "    for row in reader:\n",
        "        u, v = row['source'].strip(), row['target'].strip()\n",
        "        cap  = int(row['capacity_mbps'])\n",
        "        G.add_node(u, node_type=row['node_type_source'].strip())\n",
        "        G.add_node(v, node_type=row['node_type_target'].strip())\n",
        "        G.add_edge(u, v, capacity=cap,\n",
        "                   distance_m=int(row['distance_m']),\n",
        "                   cable_cost_tl=int(row['cable_cost_tl']))\n",
        "        if not G.has_edge(v, u):\n",
        "            G.add_edge(v, u, capacity=cap,\n",
        "                       distance_m=int(row['distance_m']),\n",
        "                       cable_cost_tl=int(row['cable_cost_tl']))\n",
        "\n",
        "print(f'Nodes : {G.number_of_nodes()}')\n",
        "print(f'Edges : {G.number_of_edges()} directed arcs')\n",
        "print(f'Nodes : {list(G.nodes())}')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 3 · Add the Super-Sink\n",
        "\n",
        "NetworkX's max-flow functions require **exactly one source and one sink**.\n",
        "We have two sink nodes (`student_labs_A`, `student_labs_B`), so we create\n",
        "a virtual `super_sink` node and connect both labs to it with infinite capacity.\n",
        "\n",
        "This lets the algorithm treat both lab blocks as a single aggregate demand point\n",
        "without artificially constraining the merge link."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "SOURCE     = 'data_center'\n",
        "SINK_NODES = ['student_labs_A', 'student_labs_B']\n",
        "SUPER_SINK = 'super_sink'\n",
        "\n",
        "G.add_node(SUPER_SINK, node_type='virtual')\n",
        "for lab in SINK_NODES:\n",
        "    G.add_edge(lab, SUPER_SINK, capacity=float('inf'))\n",
        "\n",
        "print(f'Super-sink added.  Total nodes now: {G.number_of_nodes()}')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 4 · Solve Maximum Flow (Edmonds-Karp)\n",
        "\n",
        "We call `nx.maximum_flow()` with the Edmonds-Karp flow function.\n",
        "\n",
        "**Returns:**\n",
        "- `flow_value` — total Mbps deliverable from source to sink\n",
        "- `flow_dict`  — dictionary `{u: {v: flow_on_uv}}` for every arc"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "flow_value, flow_dict = nx.maximum_flow(\n",
        "    G, SOURCE, SUPER_SINK,\n",
        "    flow_func=nx.algorithms.flow.edmonds_karp\n",
        ")\n",
        "\n",
        "print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')\n",
        "print(f'  MAXIMUM FLOW  :  {int(flow_value):,} Mbps  ({flow_value/1000:.1f} Gbps)')\n",
        "print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 5 · Inspect Per-Edge Flow Allocation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "rows = []\n",
        "for u in sorted(flow_dict):\n",
        "    for v, flow in sorted(flow_dict[u].items()):\n",
        "        if 'super_sink' in (u, v):\n",
        "            continue\n",
        "        cap = G[u][v].get('capacity', float('inf'))\n",
        "        if cap == float('inf'):\n",
        "            continue\n",
        "        rows.append({\n",
        "            'From': u, 'To': v,\n",
        "            'Flow (Mbps)': int(flow),\n",
        "            'Capacity (Mbps)': int(cap),\n",
        "            'Utilisation (%)': round(flow / cap * 100, 1)\n",
        "        })\n",
        "\n",
        "result_df = pd.DataFrame(rows)\n",
        "result_df = result_df.sort_values('Utilisation (%)', ascending=False)\n",
        "result_df.style.bar(subset=['Utilisation (%)'], color='#E74C3C', vmin=0, vmax=100)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 6 · Minimum Cut Analysis\n",
        "\n",
        "The **Min-Cut** tells us which edges are bottlenecks.  \n",
        "By the **Max-Flow Min-Cut Theorem**, the min-cut value equals the max-flow value."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "cut_value, (reachable, non_reachable) = nx.minimum_cut(G, SOURCE, SUPER_SINK)\n",
        "\n",
        "print(f'Min-cut value : {int(cut_value):,} Mbps  (= max flow ✓)')\n",
        "print(f'\\nSource side   : {sorted(n for n in reachable if n != SUPER_SINK)}')\n",
        "print(f'Sink side     : {sorted(n for n in non_reachable if n != SUPER_SINK)}')\n",
        "\n",
        "print('\\nBottleneck edges crossing the cut:')\n",
        "for u in reachable:\n",
        "    for v in G.successors(u):\n",
        "        if v in non_reachable:\n",
        "            cap = G[u][v].get('capacity', float('inf'))\n",
        "            if cap < float('inf'):\n",
        "                print(f'  ✦ {u}  →  {v}   [{cap:,} Mbps]')"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 7 · Visualisation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Run the full visualization from solution.py\n",
        "import sys\n",
        "sys.path.insert(0, os.path.join('..', 'src'))\n",
        "import importlib.util, types\n",
        "\n",
        "spec = importlib.util.spec_from_file_location(\n",
        "    'solution',\n",
        "    os.path.join('..', 'src', 'solution.py')\n",
        ")\n",
        "sol = importlib.util.module_from_spec(spec)\n",
        "spec.loader.exec_module(sol)\n",
        "\n",
        "fig_path = os.path.join('..', 'results', 'network_visualization.png')\n",
        "sol.visualise(G, flow_dict, reachable, SOURCE, SUPER_SINK, fig_path)\n",
        "\n",
        "from IPython.display import Image\n",
        "Image(fig_path)"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## 8 · Key Insights\n",
        "\n",
        "| Insight | Value |\n",
        "|---|---|\n",
        "| Maximum achievable throughput | **13,000 Mbps** |\n",
        "| Primary bottleneck | `data_center → engineering` (10 Gbps, 100% saturated) |\n",
        "| Secondary bottleneck | `science_letters → student_labs_A` (2.5 Gbps, 100% saturated) |\n",
        "| Idle capacity | `rectorate`, `library`, `student_affairs` corridors carry **0 flow** |\n",
        "| Upgrade recommendation | Add a second 10 GbE link: `data_center → student_labs_A/B` directly |\n",
        "\n",
        "> **Managerial takeaway:** The network has a single dominant path carrying 77% of all student-lab traffic. Redundancy and capacity upgrades on the `data_center ↔ engineering` segment should be the top capital expenditure priority for the IT directorate."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.9.0"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 4
}

print(f"Data type: {type(data)}")
print(f"Data structure: {data if isinstance(data, (str, int, float, bool)) else type(data).__name__}")


### Object Analysis

The JSON contains an object. Let's explore its structure:


In [ ]:
print(f"Object keys: {list(data.keys()) if isinstance(data, dict) else 'Not a dictionary'}")

# Analyze each key-value pair
if isinstance(data, dict):
    for key, value in data.items():
        print(f"Key: {key}, Type: {type(value)}, Value: {str(value)[:100]}{'...' if len(str(value)) > 100 else ''}")


### Raw Data

Complete JSON data structure:


In [ ]:
# Display formatted JSON
import json
print(json.dumps(data, indent=2, ensure_ascii=False))
